In [1]:
from pathlib import Path
import re

ROOT = Path.cwd()


def add_lang_switch_to_file(html_path: Path) -> bool:
    """Add a language switch <li> into the main <ul class=\"tabs\"> of one file.

    Returns True if the file was modified, False otherwise.
    """
    text = html_path.read_text(encoding="utf-8")

    # Only process files that actually have the main nav
    if '<ul class="tabs"' not in text:
        return False

    # Avoid double-inserting
    if 'class="lang-switch"' in text:
        return False

    # Locate the <ul class="tabs"> block
    ul_start = text.find('<ul class="tabs"')
    if ul_start == -1:
        return False

    ul_tag_end = text.find('>', ul_start)
    if ul_tag_end == -1:
        return False

    ul_close = text.find('</ul>', ul_tag_end)
    if ul_close == -1:
        return False

    # Derive language and counterpart path
    rel = html_path.relative_to(ROOT)
    parts = rel.parts
    if len(parts) < 2 or parts[0] not in {"cn", "en"}:
        return False

    lang = parts[0]
    other_lang = "en" if lang == "cn" else "cn"
    # Path inside language folder
    inner_rel = Path(*parts[1:])
    href = f"/{other_lang}/{inner_rel.as_posix()}"
    label = "English" if lang == "cn" else "中文"

    # Try to match indentation from the first <li> inside this <ul>
    block = text[ul_tag_end:ul_close]
    m = re.search(r"\n([ \t]+)<li", block)
    if m:
        indent = "\n" + m.group(1)
    else:
        # Fallback indentation
        indent = "\n    "

    insert = f'{indent}<li class="lang-switch"><a class="first" href="{href}">{label}</a></li>'

    new_text = text[:ul_close] + insert + text[ul_close:]
    html_path.write_text(new_text, encoding="utf-8")
    return True


def main() -> None:
    cn_root = ROOT / "cn"
    en_root = ROOT / "en"

    modified = []
    for lang_root in (cn_root, en_root):
        if not lang_root.is_dir():
            continue
        for html_path in lang_root.rglob("*.html"):
            try:
                if add_lang_switch_to_file(html_path):
                    modified.append(html_path.relative_to(ROOT).as_posix())
            except Exception as exc:  # best-effort logging
                print(f"[ERROR] {html_path}: {exc}")

    print("Modified files (with inserted language switch):")
    for p in modified:
        print(" -", p)


if __name__ == "__main__":
    main()


Modified files (with inserted language switch):
 - cn/lxwm/zlhz/index.html
 - cn/lxwm/zzzc/index.html
 - cn/lxwm/cgzh/index.html
 - cn/lxwm/jrwm/index.html
 - cn/yjxm/lwcc/index.html
 - cn/yjxm/lwcc/content/post_lwcc_20250221101056.html
 - cn/yjxm/lwcc/content/post_lwcc_20220221095811.html
 - cn/yjxm/lwcc/content/post_lwcc_20230321101723.html
 - cn/yjxm/lwcc/content/post_lwcc_20220221100524.html
 - cn/yjxm/lwcc/content/post_lwcc_20260309111405.html
 - cn/yjxm/lwcc/content/post_lwcc_20240221100930.html
 - cn/yjxm/lwcc/content/post_lwcc_20220221100458.html
 - cn/yjxm/lwcc/content/post_lwcc_20250221100711.html
 - cn/yjxm/lwcc/content/post_lwcc_20220221100212.html
 - cn/yjxm/lwcc/content/post_lwcc_20231131101345.html
 - cn/yjxm/lwcc/content/post_lwcc_20260210093839.html
 - cn/yjxm/zykt/index.html
 - cn/yjxm/zykt/content/post_zykt_1.html
 - cn/yjxm/zykt/content/post_zykt_2.html
 - cn/yjxm/yjbg/index.html
 - cn/yjxm/yjbg/content/post_yjbg_6.html
 - cn/yjxm/yjbg/content/post_yjbg_1.html
 - cn